# Imports and definitions

In [1]:
import pandas as pd
import numpy as np
import  matplotlib.pyplot as plt
import re
import progressbar
import os
import time
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import EvaluationUtils as utl

%matplotlib inline  

# Filter Results

In [2]:
def filterResults(res, runIDs):
    stats = {}
    i = 0
    bar = progressbar.ProgressBar(maxval=len(runIDs)).start()
    time.sleep(1)
    for runID in runIDs:
        
        corruptionsPerformed = res.loc[(res["runID"] == runID)
                                    & (res["name"] == CORRUPTIONS_STATISTIC_NAME)
                                    & (res["type"] == "vector")]

        
        droppedPacketLengths = res.loc[(res["runID"] == runID)
                                    # & (~res["module"].str.contains("sink").fillna(False))
                                    & (res["module"].isin(DROPPED_PACKET_MODULE_NAMES))
                                    & (res["name"] == DROPPED_PACKET_STATISTIC_NAME)
                                    & (res["type"] == "vector")]
        
        iterationVars = res.loc[(res["runID"] == runID)
                                & (res["attrname"] == "iterationvars")]
        
        e2eLatency = res.loc[(res["runID"] == runID)
                                & (res["module"] == E2E_LATENCY_MODULE_NAME)
                                & (res["name"] == E2E_LATENCY_STATISTIC_NAME)
                                & (res["type"] == "vector")]
                             
        stats[runID] = {}
        stats[runID]["corruptionStatisticName"] = CORRUPTIONS_STATISTIC_NAME
        stats[runID]["droppedPacketModuleNames"] = DROPPED_PACKET_MODULE_NAMES
        stats[runID]["droppedPacketStatisticName"] = DROPPED_PACKET_STATISTIC_NAME
        stats[runID]["e2eLatencyModuleName"] = E2E_LATENCY_MODULE_NAME
        stats[runID]["e2eLatencyStatisticName"] = E2E_LATENCY_STATISTIC_NAME
        
        if corruptionsPerformed.empty:
            print(f"No corruptions found: {runID}")
            if not iterationVars.empty:
                if not pd.isnull(iterationVars["attrvalue"].iloc[0]):
                    stats[runID]["iterationVars"] = iterationVars["attrvalue"].iloc[0]
            stats[runID]["corruptions"] = 0
            if not droppedPacketLengths.empty:
                stats[runID]["drops"] = 0
                for vectimes in droppedPacketLengths["vectime"]:
                    stats[runID]["drops"] += len(vectimes)
            else:
                stats[runID]["drops"] = 0
            stats[runID]["meanE2ELatency"] = e2eLatency["vecvalue"].iloc[0].mean()
            stats[runID]["singleDrops"] = 0
            stats[runID]["multiDrops"] = 0
            stats[runID]["dropDetections"] = 0
            stats[runID]["singleDropDetections"] = 0
            stats[runID]["multiDropDetections"] = 0
            stats[runID]["DropTruePositives"] = 0
            stats[runID]["DropFalseNegatives"] = 0
            stats[runID]["DropRecall"] = 0
            stats[runID]["Notes"] = "No corruptions found"
            continue
        elif len(corruptionsPerformed) > 1:
            print(f"Multiple corruptions found: {runID}")
            continue

        if droppedPacketLengths.empty:
            print(f"No dropped packets found: {runID}")
            #continue
        elif len(droppedPacketLengths) > 1:
            print(f"Multiple dropped packets found: {runID}")
            #continue
            
        
        corruptionCount = 0
        dropCount = 0
        singleDropCount = 0
        multiDropCount = 0
        singleDropDetectionCount = 0
        multiDropDetectionCount = 0
        dropDetectionCount = 0
        for T_corruption in corruptionsPerformed["vectime"].iloc[0]:
            corruptionCount += 1
            drops = 0
            if not droppedPacketLengths.empty:
                for i in range(len(droppedPacketLengths)):
                    for T_drop in droppedPacketLengths["vectime"].iloc[i]:
                        if T_drop < T_corruption:
                            continue
                        elif T_drop > (T_corruption + MAX_DETECTION_WINDOW_SECONDS):
                            break
                        else:
                            drops += 1
            if drops > 0:
                dropCount += drops
                dropDetectionCount += 1
            if drops == 1:
                singleDropCount += drops
                singleDropDetectionCount += 1
            elif drops > 1:
                multiDropCount += drops
                multiDropDetectionCount += 1
            if not iterationVars.empty:
                if not pd.isnull(iterationVars["attrvalue"].iloc[0]):
                    stats[runID]["iterationVars"] = iterationVars["attrvalue"].iloc[0]
            stats[runID]["meanE2ELatency"] = e2eLatency["vecvalue"].iloc[0].mean()
            stats[runID]["corruptions"] = corruptionCount
            stats[runID]["drops"] = dropCount
            stats[runID]["singleDrops"] = singleDropCount
            stats[runID]["multiDrops"] = multiDropCount
            stats[runID]["dropDetections"] = dropDetectionCount
            stats[runID]["singleDropDetections"] = singleDropDetectionCount
            stats[runID]["multiDropDetections"] = multiDropDetectionCount
            if multiDropDetectionCount > 0:
                stats[runID]["multiDropsPerMultiDropDetection"] = multiDropCount / multiDropDetectionCount
            stats[runID]["DropTruePositives"] = dropDetectionCount
            stats[runID]["DropFalseNegatives"] = corruptionCount - dropDetectionCount
            stats[runID]["DropRecall"] = dropDetectionCount / (dropDetectionCount + (corruptionCount - dropDetectionCount))
            stats[runID]["Notes"] = "Dropped multiple flows" if len(droppedPacketLengths) > 1 else ""
        i += 1
        bar.update(i)
    bar.finish()
    return stats
			

# Summarize stats
Repetitions of run configurations can be summed up and the recall has to be recalculated

In [3]:
def summarizeStats(stats):
    stats_summary = {}
    for runID,vals in stats.items():
        runConfig = runID.split('-')[0]
        if runConfig in stats_summary:
            stats_summary[runConfig]["corruptions"] += vals["corruptions"]
            stats_summary[runConfig]["drops"] += vals["drops"]
            stats_summary[runConfig]["singleDrops"] += vals["singleDrops"]
            stats_summary[runConfig]["multiDrops"] += vals["multiDrops"]
            stats_summary[runConfig]["dropDetections"] += vals["dropDetections"]
            stats_summary[runConfig]["singleDropDetections"] += vals["singleDropDetections"]
            stats_summary[runConfig]["multiDropDetections"] += vals["multiDropDetections"]
            if "multiDropsPerMultiDropDetection" in stats_summary[runConfig]:
                if "multiDropsPerMultiDropDetection" in vals:
                    stats_summary[runConfig]["multiDropsPerMultiDropDetection"] += vals["multiDropsPerMultiDropDetection"]
            stats_summary[runConfig]["DropTruePositives"] += vals["DropTruePositives"]
            stats_summary[runConfig]["DropFalseNegatives"] += vals["DropFalseNegatives"]
            stats_summary[runConfig]["DropRecall"] += vals["DropRecall"]
            stats_summary[runConfig]["num_repetitions"] += 1
            if vals["Notes"] != "" and stats_summary[runConfig]["Notes"] == "":
                stats_summary[runConfig]["Notes"] += vals["Notes"]
    
        else:
            stats_summary[runConfig] = vals.copy()
            stats_summary[runConfig]["num_repetitions"] = 1
            
    # Average for all values
    for runConfig, vals in stats_summary.items():
        num_rep = vals["num_repetitions"]
        del vals["Notes"]
        for key, val in vals.items():
            # if val is a number
            if isinstance(val, (int, float)):
                vals[key] = val/num_rep
        vals["num_repetitions"]=num_rep
    
    runConfigs = list(stats_summary.keys())
    return stats_summary, runConfigs

# Write Stats

In [4]:
def writeStats(stats, stats_summary, runIDs, runConfigs):
    #sort stats for iterationVars
    runIDs = sorted(runIDs, key=lambda x: stats[x]["iterationVars"] if "iterationVars" in stats[x] else x)
    i = 0
    bar = progressbar.ProgressBar(maxval=len(runIDs)*2 + len(runConfigs)).start()
    time.sleep(1)
    file = open("evaluation_results_" + time.strftime("%Y%m%d-%H%M%S") + ".txt", "w")
    file.write("### Statistic Sources ###\n")
    first_runID = next(iter(stats))
    file.write(stats[first_runID]["corruptionStatisticName"] + "\n")
    file.write(str(stats[first_runID]["droppedPacketModuleNames"]) + " - " + stats[first_runID]["droppedPacketStatisticName"] + "\n")
    file.write(stats[first_runID]["e2eLatencyModuleName"] + " - " + stats[first_runID]["e2eLatencyStatisticName"] + "\n")
    file.write("\n")
    file.write("### Drop Detection Performance Table (Summarized)###\n")
    file.write("{:<60} {:<20} {:<20} {:<20} {:<20}\n".format('simulation', 'avg corruptions', ' avg TruePositives', 'avg FalseNegatives', 'avg Recall'))
    for runConfig in runConfigs:
        file.write("{:<60} {:<20} {:<20} {:<20} {:<20}".format(runConfig, stats_summary[runConfig]["corruptions"], stats_summary[runConfig]["DropTruePositives"], stats_summary[runConfig]["DropFalseNegatives"], stats_summary[runConfig]["DropRecall"]))
        file.write("  ->  " + str(round(stats_summary[runConfig]["DropTruePositives"])) + " / " + str(round(stats_summary[runConfig]["DropFalseNegatives"])) + " / " + str(round(stats_summary[runConfig]["DropRecall"], 2)) + "\n")
        i += 1
        bar.update(i)
    file.write("\n")
    file.write("### Drop Detection Performance Table ###\n")
    file.write("{:<80} {:<20} {:<20} {:<20} {:<20} {:<20} {:<20}\n".format('simulation', 'corruptions', 'drops', 'TruePositives', 'FalseNegatives', 'Recall', 'Notes'))
    for runID in runIDs:
        runName = runID + " (" + stats[runID]["iterationVars"] + ")" if "iterationVars" in stats[runID] else runID
        file.write("{:<80} {:<20} {:<20} {:<20} {:<20} {:<20} {:<20}".format(runName, stats[runID]["corruptions"], stats[runID]["drops"], stats[runID]["DropTruePositives"], stats[runID]["DropFalseNegatives"], stats[runID]["DropRecall"], stats[runID]["Notes"]))
        file.write("  ->  " + str(stats[runID]["DropTruePositives"]) + " / " + str(stats[runID]["DropFalseNegatives"]) + " / " + str(round(stats[runID]["DropRecall"], 2)) + "\n")
        i += 1
        bar.update(i)
    file.write("\n")
    file.write("### Details ###\n")
    for runID in runIDs:
        file.write(runID + " -> " + str(stats[runID]) + "\n")
        i += 1
        bar.update(i)
    file.close()
    bar.finish()

# Plot Heatmap

In [5]:
def plotHeatMap(title, heatmap_data, xlabel, ylabel, zlabel, cmap='Reds'):
	# Extract x, y, and z values from heatmap_data
	x = [data[0] for data in heatmap_data]
	y = [data[1] for data in heatmap_data]
	z = [data[2] for data in heatmap_data]

	# Convert x, y, and z to numpy arrays
	x = np.array(x)
	y = np.array(y)
	z = np.array(z)

	# Create a meshgrid for x and y
	X, Y = np.meshgrid(np.unique(x), np.unique(y))

	# Initialize Z with zeros
	Z = np.zeros_like(X, dtype=float)

	# Create a dictionary to map (x, y) pairs to z values
	z_dict = {(x[i], y[i]): z[i] for i in range(len(z))}

	# Fill Z with the corresponding z values
	for i in range(X.shape[0]):
		for j in range(X.shape[1]):
			Z[i, j] = z_dict.get((X[i, j], Y[i, j]), 0)

	# Plot the heatmap
	plt.figure(figsize=(10, 8))
	plt.imshow(Z, cmap=cmap, interpolation='nearest', origin='lower')
	plt.colorbar(label=zlabel)
	plt.title(title)
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.xticks(range(len(np.unique(x))), np.unique(x))
	plt.yticks(range(len(np.unique(y))), np.unique(y))

	# Add values to the heatmap
	for i in range(Z.shape[0]):
		for j in range(Z.shape[1]):
			plt.text(j, i, Z[i, j], ha='center', va='center', color='black')

	plt.show()

# General Configuration

In [6]:
MAX_DETECTION_WINDOW_SECONDS = 0.001
CORRUPTIONS_STATISTIC_NAME = "corruptionsPerformed:vector"
DROPPED_PACKET_STATISTIC_NAME = "droppedPacketLengths:vector"
E2E_LATENCY_STATISTIC_NAME = "meanBitLifeTimePerPacket:vector"

# Evaluation QCI

In [19]:
DROPPED_PACKET_MODULE_NAMES = [
	"Micro.switch1.bridging.streamFilter.ingress.gateFilter[3].filter[0]",
	"Micro.switch1.bridging.streamFilter.ingress.flowFilter[1]"
	]
E2E_LATENCY_MODULE_NAME = "Micro.node2.app[1].sink"
RESULT_FILES_FILTER = "Corrupt_*.vec"

In [ ]:
res, runIDs = utl.readResultFiles(RESULT_FILES_FILTER, [CORRUPTIONS_STATISTIC_NAME, DROPPED_PACKET_STATISTIC_NAME, E2E_LATENCY_STATISTIC_NAME])
if len(res) > 0:
	stats = filterResults(res, runIDs)
	stats_summary, runConfigs = summarizeStats(stats)
	writeStats(stats, stats_summary, runIDs, runConfigs)
else:
	print("No results found")

## Stream 2

In [309]:
DROPPED_PACKET_MODULE_NAMES = [
	"Micro.switch1.bridging.streamFilter.ingress.flowFilter[2]"
	]
E2E_LATENCY_MODULE_NAME = "Micro.node2.app[4].sink"

In [ ]:
res, runIDs = utl.readResultFiles(RESULT_FILES_FILTER, [CORRUPTIONS_STATISTIC_NAME, DROPPED_PACKET_STATISTIC_NAME, E2E_LATENCY_STATISTIC_NAME])
if len(res) > 0:
	stats = filterResults(res, runIDs)
	stats_summary, runConfigs = summarizeStats(stats)
	writeStats(stats, stats_summary, runIDs, runConfigs)
else:
	print("No results found")

# Evaluation ATS
This evaluation has to take into account that there are multiple ATS Parameters that may influence how sensitive to misbehaviour the system is. 

In [311]:
DROPPED_PACKET_MODULE_NAMES = [
	"Micro.switch1.bridging.streamFilter.ingress.flowFilter[1]",
	"Micro.switch1.bridging.streamFilter.ingress.gateFilter[3].filter[0]"
	]
E2E_LATENCY_MODULE_NAME = "Micro.node2.app[1].sink"
# RESULT_FILES_FILTER = "CorruptATS_InjectionAtFront*.vec"
RESULT_FILES_FILTER = "Baseline_ATS_S2*.vec"

In [ ]:
res_ats, runIDs_ats = utl.readResultFiles(RESULT_FILES_FILTER, [CORRUPTIONS_STATISTIC_NAME, DROPPED_PACKET_STATISTIC_NAME, E2E_LATENCY_STATISTIC_NAME])
if len(res_ats) > 0:
	stats_ats = filterResults(res_ats, runIDs_ats)
	stats_summary_ats,runConfigs_ats = summarizeStats(stats_ats)
	writeStats(stats_ats, stats_summary_ats, runIDs_ats, runConfigs_ats)
else:
	print("No results found")

In [ ]:
# Extract data for heatmaps
heatmap_data_drops = []
heatmap_data_meanE2ELatency = []

for runID in runIDs_ats:
	maxResT = stats_ats[runID]["iterationVars"].split(",")[0].split("=")[1]
	maxBurst = stats_ats[runID]["iterationVars"].split(",")[1].split("=")[1]
	drops = stats_ats[runID]["drops"]
	meanE2ELatency = int(stats_ats[runID]["meanE2ELatency"] * 1000000) # Convert to microseconds
	int(meanE2ELatency * 100000)
	heatmap_data_drops.append([maxResT, maxBurst, drops])
	heatmap_data_meanE2ELatency.append([maxResT, maxBurst, meanE2ELatency])

plotHeatMap("DropsPerConfiguration", heatmap_data_drops, "maxResT", "maxBurst", "Drops")
plotHeatMap("MeanE2ELatencyPerConfiguration", heatmap_data_meanE2ELatency, "maxResT", "maxBurst", "MeanE2ELatency", cmap='Blues')


## Stream 2

In [ ]:
DROPPED_PACKET_MODULE_NAMES = [
	"Micro.switch1.bridging.streamFilter.ingress.flowFilter[2]"
	]
E2E_LATENCY_MODULE_NAME = "Micro.node2.app[4].sink"

res_ats, runIDs_ats = utl.readResultFiles(RESULT_FILES_FILTER, [CORRUPTIONS_STATISTIC_NAME, DROPPED_PACKET_STATISTIC_NAME, E2E_LATENCY_STATISTIC_NAME])
if len(res_ats) > 0:
	stats_ats = filterResults(res_ats, runIDs_ats)
	stats_summary_ats,runConfigs_ats = summarizeStats(stats_ats)
	writeStats(stats_ats, stats_summary_ats, runIDs_ats, runConfigs_ats)

	# Extract data for heatmaps
	heatmap_data_drops = []
	heatmap_data_meanE2ELatency = []

	for runID in runIDs_ats:
		maxResT = stats_ats[runID]["iterationVars"].split(",")[0].split("=")[1]
		maxBurst = stats_ats[runID]["iterationVars"].split(",")[1].split("=")[1]
		drops = stats_ats[runID]["drops"]
		meanE2ELatency = int(stats_ats[runID]["meanE2ELatency"] * 1000000) # Convert to microseconds
		int(meanE2ELatency * 100000)
		heatmap_data_drops.append([maxResT, maxBurst, drops])
		heatmap_data_meanE2ELatency.append([maxResT, maxBurst, meanE2ELatency])

	plotHeatMap("DropsPerConfiguration", heatmap_data_drops, "maxResT", "maxBurst", "Drops")
	plotHeatMap("MeanE2ELatencyPerConfiguration", heatmap_data_meanE2ELatency, "maxResT", "maxBurst", "MeanE2ELatency", cmap='Blues')

else:
	print("No results found")

# Details based on ATS parameters

In [37]:
DROPPED_PACKET_MODULE_NAMES = [
	"Micro.switch1.bridging.streamFilter.ingress.flowFilter[1]",
	"Micro.switch1.bridging.streamFilter.ingress.gateFilter[3].filter[0]"
	]
E2E_LATENCY_MODULE_NAME = "Micro.node2.app[1].sink"
RESULT_FILES_FILTER = "CorruptATS_InjectionAtFront*.vec"

In [ ]:
res_ats_d, runIDs_ats_d = utl.readResultFiles(RESULT_FILES_FILTER, [CORRUPTIONS_STATISTIC_NAME, DROPPED_PACKET_STATISTIC_NAME, E2E_LATENCY_STATISTIC_NAME])
stats_ats_d = filterResults(res_ats_d, runIDs_ats_d)

In [ ]:
# sort the results by run number (from key): same run number should be same parameters, this only works now, because I did not change them.
# TODO: get parameters in filterResults function
aggr_stats_ats = {}
for key, values in stats_ats_d.items():
    runnum = key.split('-')[1]
    if runnum in aggr_stats_ats:
        aggr_stats_ats[runnum]['data'][key] = values
        aggr_stats_ats[runnum]['runIDs'].append(key)
    else:
        aggr_stats_ats[runnum] = {'data':{key: values}, 
                                 'runIDs': [key]}

for runnum, vals in aggr_stats_ats.items():
    
    stats_summary_ats_d,runConfigs_ats_d = summarizeStats(vals['data'])
    writeStats(vals['data'], stats_summary_ats_d, vals['runIDs'], runConfigs_ats_d)
        